# Learning Objectives

- Build an LLM assistant for document-based Q&A using retrieval-augmented generation.

Ensure the library chromadb is latest. Run the below command

In [1]:
import time
import chromadb
import os
from langchain_chroma import Chroma

from langchain_core.documents import Document

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader

from datetime import datetime
# from google.colab import userdata


c:\Users\Pranay Gupta\Desktop\Agile Ventures Assignments\AG0852_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Below is the api key, endpoint, api version, deployment name for the chat model

In [18]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [19]:
from groq import Groq

client = Groq()

In [20]:
model_name = 'openai/gpt-oss-120b'

 Instantiating the embedding model

In [2]:
### Before running the command ensure the numpy version is 2.3.4. If not then upgrade numpy as per previous steps in notebook.
### Then do not restart the notebook. If you restart then you will loose all loaded variables. Just click Cancel instad of Restart

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Pranay Gupta\AppData\Local\Temp\ipykernel_24556\1865801256.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


# Load the Vector Database

In [3]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [4]:
chromadb_client

In [5]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [6]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [7]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [8]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000151C2140810>, search_kwargs={'k': 5})

# RAG Q&A

## Prompt Design

The RAG system message should clearly communicate to the LLM that the input will include a user query along with the necessary context information to address that query. Additionally, the response should rely solely on the context information provided.

In [9]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [10]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

## Retrieving relevant documents

In [11]:
user_query = "What was the automotive revenue in 2021?"

In [12]:
relevant_document_chunks = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [13]:
len(relevant_document_chunks)

5

We can inspect the first document like so:

In [14]:
for document in relevant_document_chunks:
    print(document)
    break


page_content='2022
2021
$
%
$
%
Automotive	sales
$
78,509	
$
67,210	
$
44,125	
$
11,299	
17	
%
$
23,085	
52	
%
Automotive	regulatory	credits
1,790	
1,776	
1,465	
14	
1	
%
311	
21	
%
Automotive	leasing
2,120	
2,476	
1,642	
(356)
(14)
%
834	
51	
%
Total	automotive	revenues
82,419	
71,462	
47,232	
10,957	
15	
%
24,230	
51	
%
Services	and	other
8,319	
6,091	
3,802	
2,228	
37	
%
2,289	
60	
%
Total	automotive	&	services	and	other	segment
revenue
90,738	
77,553	
51,034	
13,185	
17	
%
26,519	
52	
%
Energy	generation	and	storage	segment	revenue
6,035	
3,909	
2,789	
2,126	
54	
%
1,120	
40	
%
Total	revenues
$
96,773	
$
81,462	
$
53,823	
$
15,311	
19	
%
$
27,639	
51	
%
Automotive	&	Services	and	Other	Segment
Automotive	sales	revenue	includes	revenues	related	to	cash	and	financing	deliveries	of	new	Model	S,	Model	X,	Semi,	Model	3,	Model	Y,	and
Cybertruck	vehicles,	including	access	to	our	FSD	Capability	features	and	their	ongoing	maintenance,	internet	connectivity,	free	Supercharging	programs
and	ov

In [15]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
2022
2021
$
%
$
%
Automotive sales
$
78,509 
$
67,210 
$
44,125 
$
11,299 
17 
%
$
23,085 
52 
%
Automotive regulatory credits
1,790 
1,776 
1,465 
14 
1 
%
311 
21 
%
Automotive leasing
2,120 
2,476 
1,642 
(356)
(14)
%
834 
51 
%
Total automotive revenues
82,419 
71,462 
47,232 
10,957 
15 
%
24,230 
51 
%
Services and other
8,319 
6,091 
3,802 
2,228 
37 
%
2,289 
60 
%
Total automotive & services and other segment
revenue
90,738 
77,553 
51,034 
13,185 
17 
%
26,519 
52 
%
Energy generation and storage segment revenue
6,035 
3,909 
2,789 
2,126 
54 
%
1,120 
40 
%
Total revenues
$
96,773 
$
81,462 
$
53,823 
$
15,311 
19 
%
$
27,639 
51 
%
Automotive & Services and Other Segment
Automotive sales revenue includes revenues related to cash and financing deliveries of new Model S, Model X, Semi, Model 3, Model Y, and
Cybertruck vehicles, including access to our FSD Capability features and their ongoing maintenance, internet connectivity, free Supercharging program

## Composing the response

To compose the response to user queries, we assemble the prompt that uses the system message defined above and the dynamically retrieved context for the user query.

In [16]:
user_query = "What was the automotive revenue in 2021?"

In [21]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

The automotive revenue in 2021 was **$71,462 million**.


## Lets Use Advance RAG Techniques for retreivals:

#### 1. Query Expansion.

In query expansion, we ask the LLM to generate variations of the original user query. Then, we use each of these variations to retrieve the relevant context. The final context used is the **unique** set of documents that are retreived across all the query expansions.

In [22]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [23]:
user_input = "What was the automotive revenue in 2021?"

In [24]:
model_name = 'openai/gpt-oss-120b'

In [25]:
prompt = [
    {'role':'system','content':query_expansion_system_message},
    {'role':'user','content':user_message_template.format(
        question=user_input
    )}
]

In [26]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nWhat was the automotive revenue in 2021?\n</Question>\n'}]

In [27]:
query_expansions = client.chat.completions.create(
    model=model_name,
    messages=prompt,
    temperature=0
)

In [28]:
type(query_expansions.choices[0].message.content)

str

In [29]:
print(query_expansions)

ChatCompletion(id='chatcmpl-abc47e8c-cf9b-413e-b4db-9d4e7cac7aa5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What was the automotive revenue in 2021?\nHow much automotive revenue did the company generate in 2021?\nWhat is the 2021 automotive revenue figure?', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user wants query expansion: produce at least 3 versions of the question, each on a new line, no numbering or bullets. Should rephrase the question with synonyms, different phrasing. Keep "automotive revenue" and "2021". Could also ask "What was the revenue from automotive segment in 2021?" "How much automotive revenue did the company generate in 2021?" "What is the 2021 automotive revenue figure?" "What were the automotive sales revenues for 2021?" Provide at least 3. Ensure each line is a question. No extra text.', tool_calls=None))], created=1780421830, model='openai/gpt-oss-120b'

In [30]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce at least 3 versions of the question, each on a new line, no numbering or bullets. Should rephrase the question with synonyms, different phrasing. Keep "automotive revenue" and "2021". Could also ask "What was the revenue from automotive segment in 2021?" "How much automotive revenue did the company generate in 2021?" "What is the 2021 automotive revenue figure?" "What were the automotive sales revenues for 2021?" Provide at least 3. Ensure each line is a question. No extra text.


In [31]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [32]:
query_expansions_list 

['What was the automotive revenue in 2021?',
 'How much automotive revenue did the company generate in 2021?',
 'What is the 2021 automotive revenue figure?']

In [33]:
expanded_context_list = []

In [34]:
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

In [35]:
len(expanded_context_list)

15

In [36]:
expanded_context_list

['2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13,185\t\n17\t\n%\n26,519\t\n52\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\trevenue\n6,035\t\n3,909\t\n2,789\t\n2,126\t\n54\t\n%\n1,120\t\n40\t\n%\nTotal\trevenues\n$\n96,773\t\n$\n81,462\t\n$\n53,823\t\n$\n15,311\t\n19\t\n%\n$\n27,639\t\n51\t\n%\nAutomotive\t&\tServices\tand\tOther\tSegment\nAutomotive\tsales\trevenue\tincludes\trevenues\trelated\tto\tcash\tand\tfinancing\tdeliveries\tof\tnew\tModel\tS,\tModel\tX,\tSem

In [37]:
final_context_documents = set(expanded_context_list)

In [38]:
len(final_context_documents)

7

In [39]:
final_context_documents

{'(Dollars\tin\tmillions)\n2023\n2022\n2021\n$\n%\n$\n%\nCost\tof\trevenues\nAutomotive\tsales\n$\n65,121\t\n$\n49,599\t\n$\n32,415\t\n$\n15,522\t\n31\t\n%\n$\n17,184\t\n53\t\n%\nAutomotive\tleasing\n1,268\t\n1,509\t\n978\t\n(241)\n(16)\n%\n531\t\n54\t\n%\nTotal\tautomotive\tcost\tof\trevenues\n66,389\t\n51,108\t\n33,393\t\n15,281\t\n30\t\n%\n17,715\t\n53\t\n%\nServices\tand\tother\n7,830\t\n5,880\t\n3,906\t\n1,950\t\n33\t\n%\n1,974\t\n51\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\ncost\tof\trevenues\n74,219\t\n56,988\t\n37,299\t\n17,231\t\n30\t\n%\n19,689\t\n53\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\n4,894\t\n3,621\t\n2,918\t\n1,273\t\n35\t\n%\n703\t\n24\t\n%\nTotal\tcost\tof\trevenues\n$\n79,113\t\n$\n60,609\t\n$\n40,217\t\n$\n18,504\t\n31\t\n%\n$\n20,392\t\n51\t\n%\nGross\tprofit\ttotal\tautomotive\n$\n16,030\t\n$\n20,354\t\n$\n13,839\t\nGross\tmargin\ttotal\tautomotive\n19.4\t\n%\n28.5\t\n%\n29.3\t\n%\nGross\tprofit\ttotal\tautomotive\t&\tservices\tand\tothe

In [40]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [41]:
content_for_query= "\n\n--\n\n".join(
    map(str,final_context_documents)
)

In [42]:
prompts = [
    {'role':'system','content':qna_system_message},
    {'role':'user','content':qna_user_message_template.format(
        context = content_for_query,
        question=user_input
    )}
]

In [43]:
prompts

[{'role': 'system',
  'content': '\nYou are an assistant to a financial services firm who answers user queries on annual reports.\nUser input will have the context required by you to answer user queries.\nThis context will be delimited by: <Context> and </Context>.\nThe context contains references to specific portions of a document relevant to the user query.\n\nUser queries will be delimited by: <Question> and </Question>.\n\nPlease answer user queries only using the context provided in the input.\nDo not mention anything about the context in your final answer. Your response should only contain the answer to the question.\n\nIf the answer is not found in the context, respond "I don\'t know".\n'},
 {'role': 'user',
  'content': '\n<Context>\nHere are some documents that are relevant to the question mentioned below.\nfeature\trelease\tin\tNorth\tAmerica.\n\t\nAutomotive\tregulatory\tcredits\trevenue\tincreased\t$311\tmillion,\tor\t21%,\tin\tthe\tyear\tended\tDecember\t31,\t2022\tas\tcom

In [44]:
try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompts,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

The automotive revenue in 2021 was $47.232 billion.
